In [2]:
import ee
import xarray
import rioxarray
import rasterio
from rasterio.transform import from_origin
import threading
import warnings
import dask.dataframe as dd

In [3]:
ee.Initialize(opt_url='https://earthengine-highvolume.googleapis.com')

In [4]:
def prep_sr_l8(image):
    # Develop masks for unwanted pixels (fill, cloud, cloud shadow).
    qa_mask = image.select('QA_PIXEL').bitwiseAnd(int('11111', 2)).eq(0)
    saturation_mask = image.select('QA_RADSAT').eq(0)

    # Apply the scaling factors to the appropriate bands.
    def get_factor_img(factor_names):
        factor_list = image.toDictionary().select(factor_names).values()
        return ee.Image.constant(factor_list)
    
    scale_img = get_factor_img([
        'REFLECTANCE_MULT_BAND_.|TEMPERATURE_MULT_BAND_ST_B10'])
    offset_img = get_factor_img([
        'REFLECTANCE_ADD_BAND_.|TEMPERATURE_ADD_BAND_ST_B10'])
    scaled = image.select('SR_B.|ST_B10').multiply(scale_img).add(offset_img)

    # Replace original bands with scaled bands and apply masks.
    return image.addBands(scaled, None, True)\
        .updateMask(qa_mask).updateMask(saturation_mask)



In [1]:
from dask.distributed import Client
client = Client()
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: http://127.0.0.1:8787/status,
Dashboard: http://127.0.0.1:8787/status,Workers: 4
Total threads: 16,Total memory: 31.92 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:57689,Workers: 4
Dashboard: http://127.0.0.1:8787/status,Total threads: 16
Started: Just now,Total memory: 31.92 GiB
Comm: tcp://127.0.0.1:57718,Total threads: 4
Dashboard: http://127.0.0.1:57723/status,Memory: 7.98 GiB
Nanny: tcp://127.0.0.1:57692,


In [11]:
client.close()

In [5]:
#CALIFORNIA = ee.FeatureCollection("projects/calfuels/assets/Boundaries/California")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU_remove_NV_remove_lake")

ic = (ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).filterBounds(LTBMU.geometry())
      .filterDate('2019-01-01', '2019-12-31'))

ic_xr = xarray.open_dataset(ic, engine = "ee", crs='EPSG:3310', scale = 30, chunks="auto")

In [9]:
ic_xr

<xarray.Dataset>
Dimensions:        (time: 828, X: 30211, Y: 35264)
Coordinates:
  * time           (time) datetime64[ns] 2019-01-01T18:09:39.941000 ... 2019-...
  * X              (X) float64 -4.186e+05 -4.186e+05 ... 4.877e+05 4.877e+05
  * Y              (Y) float64 -5.992e+05 -5.992e+05 ... 4.586e+05 4.587e+05
Data variables: (12/20)
    SR_B1          (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    SR_B2          (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    SR_B3          (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    SR_B4          (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    SR_B5          (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    SR_B6          (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    ...             ...
    ST_QA          (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    ST_TRAD        (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    ST_URAD        (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    QA_PIXEL       (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    QA_RADSAT      (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
    NDVI           (time, X, Y) float32 dask.array<chunksize=(48, 512, 256), meta=np.ndarray>
Attributes: (12/26)
    date_range:             [1365638400000, 1654560000000]
    description:            <p>This dataset contains atmospherically correcte...
    keywords:               ['cfmask', 'cloud', 'fmask', 'global', 'l8sr', 'l...
    period:                 0
    product_tags:           ['global', 'sr', 'reflectance', 'l8sr', 'lst', 'c...
    provider:               USGS
    ...                     ...
    visualization_1_name:   Near Infrared (543)
    visualization_2_bands:  SR_B7,SR_B5,SR_B3
    visualization_2_max:    30000.0
    visualization_2_min:    0.0
    visualization_2_name:   Shortwave Infrared (753)
    crs:                    EPSG:3310

In [6]:
def calculate_ndvi(red, nir):
    return (nir - red) / (nir + red)

In [7]:
ndvi = (ic_xr['SR_B5'] - ic_xr['SR_B4']) / (ic_xr['SR_B5'] - ic_xr['SR_B4'])

In [11]:
ndvi = ic_xr['SR_B5'] - ic_xr['SR_B4']

In [12]:
ic_xr['NDVI'] = ndvi

In [13]:
ic_xr_result = ic_xr.compute()

AttributeError: 'ImageCollection' object has no attribute 'toList'

In [14]:
def calculate_ndvi(red, nir):
    return (nir - red) / (nir + red)

# Apply the custom function to compute NDVI
ndvi = xarray.apply_ufunc(
    calculate_ndvi,
    ic_xr['SR_B4'],  # Red band variable
    ic_xr['SR_B5'],  # Near-Infrared band variable
    dask='parallelized',  # Use dask if you want to handle large datasets
    input_core_dims=[[], []],  # The dimensions of the input variables
    output_core_dims=[[]],  # The dimension of the output variable
    vectorize=True,  # Whether to vectorize the function
    output_dtypes=[float],  # The dtype of the output variable
)



In [25]:
ic_xr['NDVI'] = ndvi
ic_xr = ic_xr.rename({'X': 'x', 'Y': 'y'})
ic_xr = ic_xr.transpose('time', 'y', 'x')

In [26]:
print(ic_xr)

<xarray.Dataset>
Dimensions:        (time: 828, x: 30211, y: 35264)
Coordinates:
  * time           (time) datetime64[ns] 2019-01-01T18:09:39.941000 ... 2019-...
  * x              (x) float32 -4.186e+05 -4.186e+05 ... 4.877e+05 4.877e+05
  * y              (y) float32 4.587e+05 4.586e+05 ... -5.992e+05 -5.992e+05
Data variables: (12/20)
    SR_B1          (time, y, x) float32 dask.array<chunksize=(48, 256, 512), meta=np.ndarray>
    SR_B2          (time, y, x) float32 dask.array<chunksize=(48, 256, 512), meta=np.ndarray>
    SR_B3          (time, y, x) float32 dask.array<chunksize=(48, 256, 512), meta=np.ndarray>
    SR_B4          (time, y, x) float32 dask.array<chunksize=(48, 256, 512), meta=np.ndarray>
    SR_B5          (time, y, x) float32 dask.array<chunksize=(48, 256, 512), meta=np.ndarray>
    SR_B6          (time, y, x) float32 dask.array<chunksize=(48, 256, 512), meta=np.ndarray>
    ...             ...
    ST_QA          (time, y, x) float32 dask.array<chunksize=(48, 256, 5

In [ ]:
print(ic_xr.isel(time=0))

In [29]:
ic_xr["NDVI"].isel(time=0).rio.to_raster("ndvi.tif")

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [12]:
df = ic_xr.to_dask_dataframe()

c:\Users\knigh\anaconda3\envs\grad\Lib\site-packages\IPython\core\interactiveshell.py:3517: PerformanceWarning: Reshaping is producing a large chunk. To accept the large
chunk and silence this warning, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': False}):
    ...     array.reshape(shape)

To avoid creating the large chunks, set the option
    >>> with dask.config.set(**{'array.slicing.split_large_chunks': True}):
    ...     array.reshape(shape)Explictly passing ``limit`` to ``reshape`` will also silence this warning
    >>> array.reshape(shape, limit='128 MiB')
  if await self.run_code(code, result, async_=asy):


In [ ]:
df.map_partitions(len).compute()

In [13]:
print(df)

Dask DataFrame Structure:
                          time        X        Y    SR_B1    SR_B2    SR_B3    SR_B4    SR_B5    SR_B6    SR_B7 SR_QA_AEROSOL   ST_B10 ST_ATRAN ST_CDIST  ST_DRAD  ST_EMIS  ST_EMSD    ST_QA  ST_TRAD  ST_URAD QA_PIXEL QA_RADSAT
npartitions=36                                                                                                                                                                                                                   
0               datetime64[ns]  float32  float32  float32  float32  float32  float32  float32  float32  float32       float32  float32  float32  float32  float32  float32  float32  float32  float32  float32  float32   float32
32090240                   ...      ...      ...      ...      ...      ...      ...      ...      ...      ...           ...      ...      ...      ...      ...      ...      ...      ...      ...      ...      ...       ...
...                        ...      ...      ...      ...      ...    

In [8]:
# Define a function to calculate NDVI
def calculate_ndvi(row):
    red = row['SR_B4']
    nir = row['SR_B5']
    ndvi = (nir - red) / (nir + red)
    return ndvi

In [14]:
# Define a function to calculate NDVI
def calculate_ndvi(red_col, nir_col):
    ndvi = (nir_col - red_col) / (nir_col + red_col)
    return ndvi

In [10]:
df['NDVI'] = df.apply(calculate_ndvi, axis=1, meta=('ndvi', 'f8'))

In [15]:
df['NDVI'] = dd.map_partitions(calculate_ndvi, df['SR_B4'], df['SR_B5'])

In [19]:
df = df.compute()

ConnectionError: ('Connection aborted.', RemoteDisconnected('Remote end closed connection without response'))

In [ ]:
# Lazy operation! Does not load the raster into memory!!
da = xarray.open_dataset("https://github.com/mapbox/rasterio/raw/1.2.1/tests/data/RGB.byte.tif")

In [ ]:
print(da)

In [12]:
print(ic_xr.coarsen.__doc__)


        Coarsen object for Datasets.

        Parameters
        ----------
        dim : mapping of hashable to int, optional
            Mapping from the dimension name to the window size.
        boundary : {"exact", "trim", "pad"}, default: "exact"
            If 'exact', a ValueError will be raised if dimension size is not a
            multiple of the window size. If 'trim', the excess entries are
            dropped. If 'pad', NA will be padded.
        side : {"left", "right"} or mapping of str to {"left", "right"}, default: "left"
        coord_func : str or mapping of hashable to str, default: "mean"
            function (name) that is applied to the coordinates,
            or a mapping from coordinate name to function (name).

        Returns
        -------
        core.rolling.DatasetCoarsen

        See Also
        --------
        core.rolling.DatasetCoarsen
        DataArray.coarsen

        :ref:`reshape.coarsen`
            User guide describing :py:func:`~xarray.D

In [ ]:
factor = 100 // 30
resample = ic_xr.coarsen(X=factor, Y=factor).mean()

In [ ]:
warnings.filterwarnings("ignore")
LTBMU = ee.FeatureCollection("projects/calfuels/assets/Boundaries/LTBMU")

#ic = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).select(['SR_B1', 'SR_B2', 'SR_B3', 'SR_B4', 'SR_B5', 'SR_B6', 'SR_B7'], 
          #['UBlue', 'Blue', 'Green', 'Red', 'NIR', 'SWIR1', 'SWIR2']).filterBounds(LTBMU.geometry())

ic_landsat = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').map(prep_sr_l8).filterBounds(LTBMU.geometry())
ic_dem = ee.ImageCollection("USGS/3DEP/1m").filterBounds(LTBMU.geometry())
ic_xr = xarray.open_mfdataset([ic_landsat, ic_dem], engine = "ee", crs='EPSG:3310', scale = 30)



In [ ]:
ds = xarray.open_mfdataset(['ee://ECMWF/ERA5_LAND/HOURLY', 'ee://NASA/GDDP-CMIP6'],
                           engine='ee', crs='EPSG:4326', scale=0.25)

In [5]:
red_band = ic_xr['SR_B4'].isel(time=0).chunk(chunks={"X": 1000, "Y": 1000})
nir_band = ic_xr['SR_B5'].isel(time=0).chunk(chunks={"X": 1000, "Y": 1000})
ndvi = (nir_band - red_band) / (nir_band + red_band)
ndvi_transpose = ndvi.transpose('Y', 'X').rename({"X": "x", "Y": "y"})

In [ ]:
print(ndvi_transpose.rio.to_raster.__doc__)

In [ ]:
print(ic_xr['SR_B5'].isel(time=0).chunk.__doc__)

In [ ]:
print(ndvi_transpose.to_netcdf.__doc__)

In [6]:
ndvi_transpose.rio.to_raster(raster_path="ndvi.tif", windowed=True, tiled=True, lock=threading.Lock())

c:\Users\knigh\anaconda3\envs\ee\Lib\site-packages\dask\core.py:127: RuntimeWarning: invalid value encountered in divide
  return func(*(_execute_task(a, cache) for a in args))


In [30]:
ic = ee.ImageCollection('ECMWF/ERA5_LAND/HOURLY').filterDate('1992-10-05', '1993-03-31')
leg1 = ee.Geometry.Rectangle(113.33, -43.63, 153.56, -10.66)
ds = xarray.open_dataset(
    ic,
    engine='ee',
    projection=ic.first().select(0).projection(),
    geometry=leg1
)

In [ ]:
ds_renamed = ds.rename({'lon': 'x', 'lat': 'y'})
ds_renamed_transpose = ds_renamed.transpose('time', 'y', 'x')
ds_renamed_transpose.isel(time=0).rio.to_raster("hourly.tif")
#ds_renamed_transpose.rio.to_raster(output_raster_path, driver='GTiff', crs='EPSG:4326', compress='lzw')

In [ ]:
'''ds_img = ds.sel(time='1992-10-05T00:00:00.000000000')
print(ds_img.coords)
lat_range = slice(-10, -20.0)  # Replace with your desired latitude range
lon_range = slice(120, 130)  # Replace with your desired longitude range

# Use .sel to filter the dataset
tiled_img = ds_img.sel(lat=lat_range, lon=lon_range)
print(tiled_img.coords)
# Print the filtered dataset
#print(tiled_img)'''